# 🐖 10 | Comparación armonizada, variación entre fechas e incertidumbre

**RECURSO: CPU (sin acelerador)**. No hace entrenamiento ni predicciones. Lee **los resultados COCOeval armonizados** del Notebook 09, estudia variaciones entre fechas, genera gráficas y Excel, y exporta un ZIP.

> **Advertencia:** solo hay **cinco fechas de test**. Las bandas bootstrap que calcularemos corresponden a la **media no ponderada de diferencias de AP entre fechas**, NO a un intervalo de confianza del mAP COCO global. La incertidumbre estimada será exploratoria e inestable por el pequeño número de fechas.

In [ ]:
%pip -q install "pandas>=2,<3" numpy openpyxl matplotlib


In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime,timezone
import json,zipfile,hashlib
import pandas as pd,numpy as np,matplotlib.pyplot as plt

drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖')
assert ROOT.is_dir(),'Carpeta del proyecto no encontrada. Comprueba la cuenta.'
RES=ROOT/'03_Resultados'
EXPORT=ROOT/'05_Exportaciones_ZIP'
REPORT=RES/'05_Reportes'/'CERDOS_09_Armonizacion_COCO.json'
EXCEL_09=RES/'03_Metricas'/'CERDOS_09_Comparacion_Armonizada_COCO.xlsx'
assert REPORT.exists() and EXCEL_09.exists(),'Completa primero el Notebook 09 con GPU T4.'
rep=json.loads(REPORT.read_text(encoding='utf8'))
assert rep.get('etapa')=='09_evaluacion_armonizada_sin_entrenamiento','Reporte 09 inesperado'
G=pd.DataFrame(rep['global']);D=pd.DataFrame(rep['por_fecha'])
assert len(G)==2 and len(D)==10 and D['fecha'].nunique()==5,'Reporte incompleto'
assert set(D.groupby('modelo')['fecha'].nunique())=={5},'Fechas de evaluación no coinciden'
print('✅ Comparación armonizada disponible:')
display(G)


## 2. Comparación apareada de AP por fecha

La **unidad de remuestreo es la fecha completa**. Esto reduce el riesgo de tratar imágenes del mismo día como independientes, pero con cinco unidades no permite conclusiones definitivas.

In [ ]:
METRICAS=['mAP50_95_mascaras','mAP50_mascaras','mAP75_mascaras']
Y='YOLO11n-seg';M='Mask R-CNN ResNet50-FPN'
P=D.pivot(index='fecha',columns='modelo',values=METRICAS)
assert not P.isna().any().any(),'Faltan mediciones para algún modelo o fecha'
rows=[];bootrows=[]
rg=np.random.default_rng(2026)
for metric in METRICAS:
    diff=(P[(metric,Y)]-P[(metric,M)]).to_numpy(dtype=float)
    dates=P.index.to_list()
    for date,val in zip(dates,diff):
        rows.append({'fecha':date,'metrica':metric,'YOLO11n_seg':float(P.loc[date,(metric,Y)]),
            'MaskRCNN':float(P.loc[date,(metric,M)]),'delta_YOLO_menos_MaskRCNN':float(val)})
    samples=rg.choice(diff,size=(10000,len(diff)),replace=True).mean(axis=1)
    lo,hi=np.quantile(samples,[.025,.975])
    g0=float(G.loc[G.modelo.eq(Y),metric].iloc[0]);g1=float(G.loc[G.modelo.eq(M),metric].iloc[0])
    bootrows.append({'metrica':metric,'n_fechas':len(diff),'delta_mAP_global_COCO':g0-g1,
        'media_no_ponderada_deltas_por_fecha':float(diff.mean()),
        'IC_exploratorio_2_5pct_media_fecha':float(lo),
        'IC_exploratorio_97_5pct_media_fecha':float(hi),
        'n_bootstrap':10000,'semilla':2026,'alcance_IC':'NO es IC del mAP global; 5 fechas'})
DIFF=pd.DataFrame(rows);BOOT=pd.DataFrame(bootrows)
print('✅ Diferencias por fecha:');display(DIFF)
print('✅ Bootstrap exploratorio agrupado por fecha:');display(BOOT)


## 3. Gráficos y Excel verificable

Diferencias positivas indican mayor AP de YOLO en esa fecha, bajo COCOeval. Las figuras y tablas no son evidencia de generalización a otras granjas.

In [ ]:
EXCEL=RES/'03_Metricas'/'CERDOS_10_Analisis_Armonizado_Fecha.xlsx'
FIG=RES/'04_Figuras'/'CERDOS_10_Diferencia_AP_por_Fecha.png'
JREPORT=RES/'05_Reportes'/'CERDOS_10_Analisis_Incertidumbre.json'
for f in [EXCEL,FIG,JREPORT]:f.parent.mkdir(parents=True,exist_ok=True)
with pd.ExcelWriter(EXCEL,engine='openpyxl') as x:
    G.to_excel(x,sheet_name='COCO_global',index=False)
    D.to_excel(x,sheet_name='COCO_por_fecha',index=False)
    DIFF.to_excel(x,sheet_name='Deltas_pareados',index=False)
    BOOT.to_excel(x,sheet_name='Bootstrap_fechas',index=False)
    pd.DataFrame({'advertencias':[
        'El test ya fue consultado en 04/07: no es una nueva evaluación externa.',
        'COCOeval común armoniza evaluador y GT, no el presupuesto de entrenamiento.',
        'Solo cinco fechas de test; los intervalos bootstrap son exploratorios.',
        'El IC estimado corresponde a la media de AP por fecha, NO a mAP global.',
        'No inferir animales únicos, comportamiento ni transferencia entre granjas.',
        'Las comparaciones por test no deben usarse para seleccionar hiperparámetros.']}).to_excel(x,sheet_name='Interpretacion',index=False)

main=DIFF.loc[DIFF.metrica.eq('mAP50_95_mascaras')].sort_values('fecha')
fig,ax=plt.subplots(figsize=(10,5.5))
ax.bar(main['fecha'],main['delta_YOLO_menos_MaskRCNN']*100)
ax.axhline(0,color='black',linewidth=1)
ax.set(ylabel='Diferencia de AP@50:95 en puntos porcentuales',xlabel='Fecha de captura',
       title='FaroPigSeg · Diferencia YOLO − Mask R-CNN por fecha · COCOeval')
ax.tick_params(axis='x',rotation=25)
fig.tight_layout();fig.savefig(FIG,dpi=200,bbox_inches='tight');plt.show();plt.close(fig)
notes=['Se comparte evaluador COCOeval y GT; NO se igualan épocas/arquitectura/hardware.',
       'El test no es nuevo. Las métricas globales ya fueron vistas en ensayos anteriores.',
       'La dispersión entre cinco fechas solo aporta evidencia exploratoria.',
       'El bootstrap reporta la media de diferencias entre fechas, no el mAP global.']
record={'fecha_utc':datetime.now(timezone.utc).isoformat(),'etapa':'10_analisis_fechas_CPU',
    'n_fechas':5,'n_imagenes':373,'tabla_bootstrap':BOOT.to_dict('records'),
    'advertencias':notes,'origen_report_09':str(REPORT)}
JREPORT.write_text(json.dumps(record,ensure_ascii=False,indent=2),encoding='utf8')
print('✅ Excel:',EXCEL,chr(10),'✅ Figura:',FIG,chr(10),'✅ Informe:',JREPORT)


## 4. Exportar ZIP de resultados (sin dataset y sin pesos)

In [ ]:
from google.colab import files
EXPORT.mkdir(exist_ok=True,parents=True)
ZIP=EXPORT/'CERDOS_10_Analisis_Comparativo_Incertidumbre.zip'
with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for f in [EXCEL,FIG,JREPORT]:
        assert f.is_file();z.write(f,arcname=f.name)
with zipfile.ZipFile(ZIP) as z:assert z.testzip() is None,'ZIP corrupto'
print('✅ ZIP guardado en Drive:',ZIP)
files.download(str(ZIP))


### Próxima etapa editorial

Solo después de revisar el ZIP 09 y 10: definir una comparación de presupuestos de entrenamiento, robustez a oclusiones, nuevas semillas y una muestra verdaderamente independiente de otra granja o sesión. **FaroPigSeg es imagen estática y no permite inferir identidades ni estabilidad temporal.** Para seguimiento de cerdos se requerirá una base de **videos** con anotaciones de individuos.